In [1]:
import sys
import pandas as pd
import numpy as np

print("Python version:", sys.version)
print("Python location:", sys.executable)
print("Pandas version:", pd.__version__)
print("NumPy version:", np.__version__)

Python version: 3.13.17 (tags/v3.13.17:64e080d, Oct 21 2026, 14:08:53) [MSC v.1944 64 bit (AMD64)]
Python location: c:\Users\priya\AppData\Local\Programs\Python\Python313\python.exe
Pandas version: 3.1.1
NumPy version: 2.5.4


In [2]:
from pathlib import Path

data_folder = Path.cwd() / "data"

print("Data folder exists:", data_folder.exists())

for file in data_folder.iterdir():
    print(file.name)

Data folder exists: True
cast.csv
crew.csv
genres.csv
movies.csv
movie_genres.csv
movie_keywords.csv


In [3]:
import pandas as pd
import numpy as np
from pathlib import Path

# Locate the project data folder
data_folder = Path.cwd() / "data"

# Load all six CSV files
movies = pd.read_csv(data_folder / "movies.csv")
genres = pd.read_csv(data_folder / "genres.csv")
movie_genres = pd.read_csv(data_folder / "movie_genres.csv")
cast = pd.read_csv(data_folder / "cast.csv")
crew = pd.read_csv(data_folder / "crew.csv")
movie_keywords = pd.read_csv(data_folder / "movie_keywords.csv")

# Organise the datasets
datasets = {
    "Movies": movies,
    "Genres": genres,
    "Movie Genres": movie_genres,
    "Cast": cast,
    "Crew": crew,
    "Movie Keywords": movie_keywords
}

print("All six datasets loaded successfully!")

All six datasets loaded successfully!


In [4]:
for name, df in datasets.items():
    print(f"{name}: {df.shape[0]} rows and {df.shape[1]} columns")

Movies: 2509 rows and 12 columns
Genres: 19 rows and 2 columns
Movie Genres: 6888 rows and 2 columns
Cast: 24917 rows and 6 columns
Crew: 7297 rows and 6 columns
Movie Keywords: 41701 rows and 3 columns


In [5]:
for name, df in datasets.items():
    print(f"\n{'='*50}")
    print(f"DATASET: {name}")
    print(f"{'='*50}")
    print(df.info())


DATASET: Movies
<class 'pandas.DataFrame'>
RangeIndex: 2509 entries, 0 to 2508
Data columns (total 12 columns):
 #   Column             Non-Null Count  Dtype  
---  ------             --------------  -----  
 0   movie_id           2509 non-null   int64  
 1   title              2509 non-null   str    
 2   original_language  2509 non-null   str    
 3   release_date       2509 non-null   str    
 4   runtime            2509 non-null   int64  
 5   budget             2509 non-null   float64
 6   revenue            2509 non-null   float64
 7   popularity         2509 non-null   float64
 8   vote_average       2509 non-null   float64
 9   vote_count         2509 non-null   int64  
 10  status             2509 non-null   str    
 11  overview           2509 non-null   str    
dtypes: float64(4), int64(3), str(5)
memory usage: 235.3 KB
None

DATASET: Genres
<class 'pandas.DataFrame'>
RangeIndex: 19 entries, 0 to 18
Data columns (total 2 columns):
 #   Column      Non-Null Count  Dtype
---

In [6]:
print("MISSING VALUE REPORT")
print("-" * 40)

for name, df in datasets.items():
    missing = df.isnull().sum().sum()
    print(f"{name}: {missing} missing values")

MISSING VALUE REPORT
----------------------------------------
Movies: 0 missing values
Genres: 0 missing values
Movie Genres: 0 missing values
Cast: 11 missing values
Crew: 0 missing values
Movie Keywords: 0 missing values


In [7]:
cast["character_name"] = cast["character_name"].fillna("Unknown")

print("Missing character names after cleaning:",
      cast["character_name"].isnull().sum())

Missing character names after cleaning: 0


In [8]:
print("DUPLICATE RECORD REPORT")
print("-" * 40)

for name, df in datasets.items():
    print(f"{name}: {df.duplicated().sum()} duplicates")

DUPLICATE RECORD REPORT
----------------------------------------
Movies: 6 duplicates
Genres: 0 duplicates
Movie Genres: 10 duplicates
Cast: 15 duplicates
Crew: 8 duplicates
Movie Keywords: 20 duplicates


In [9]:
cleaned_datasets = {}

for name, df in datasets.items():
    before = len(df)

    cleaned_df = df.drop_duplicates().copy()

    removed = before - len(cleaned_df)
    cleaned_datasets[name] = cleaned_df

    print(f"{name}: {removed} duplicate rows removed")

Movies: 6 duplicate rows removed
Genres: 0 duplicate rows removed
Movie Genres: 10 duplicate rows removed
Cast: 15 duplicate rows removed
Crew: 8 duplicate rows removed
Movie Keywords: 20 duplicate rows removed


In [11]:
movies = cleaned_datasets["Movies"]
genres = cleaned_datasets["Genres"]
movie_genres = cleaned_datasets["Movie Genres"]
cast = cleaned_datasets["Cast"]
crew = cleaned_datasets["Crew"]
movie_keywords = cleaned_datasets["Movie Keywords"]

datasets = {
    "Movies": movies,
    "Genres": genres,
    "Movie Genres": movie_genres,
    "Cast": cast,
    "Crew": crew,
    "Movie Keywords": movie_keywords
}

In [12]:
print("DUPLICATE CHECK AFTER CLEANING")
print("-" * 40)

for name, df in datasets.items():
    print(f"{name}: {df.duplicated().sum()} duplicates remaining")

DUPLICATE CHECK AFTER CLEANING
----------------------------------------
Movies: 0 duplicates remaining
Genres: 0 duplicates remaining
Movie Genres: 0 duplicates remaining
Cast: 0 duplicates remaining
Crew: 0 duplicates remaining
Movie Keywords: 0 duplicates remaining


In [13]:
movies["release_date"] = pd.to_datetime(
    movies["release_date"],
    errors="coerce"
)

print("Invalid release dates:", movies["release_date"].isna().sum())
print("Release date data type:", movies["release_date"].dtype)

Invalid release dates: 0
Release date data type: datetime64[us]


In [14]:
print("ZERO VALUE CHECK")
print("-" * 35)

print("Movies with zero budget:", (movies["budget"] == 0).sum())
print("Movies with zero revenue:", (movies["revenue"] == 0).sum())

ZERO VALUE CHECK
-----------------------------------
Movies with zero budget: 90
Movies with zero revenue: 102


In [15]:
print("MOVIE ID VALIDATION")
print("-" * 35)

print("Total movie records:", len(movies))
print("Unique movie IDs:", movies["movie_id"].nunique())
print("Duplicate movie IDs:", movies["movie_id"].duplicated().sum())

MOVIE ID VALIDATION
-----------------------------------
Total movie records: 2503
Unique movie IDs: 2503
Duplicate movie IDs: 0


In [16]:
print("RELATIONSHIP VALIDATION")
print("-" * 40)

valid_movie_ids = set(movies["movie_id"])

related_tables = {
    "Movie Genres": movie_genres,
    "Cast": cast,
    "Crew": crew,
    "Movie Keywords": movie_keywords
}

for name, df in related_tables.items():
    invalid_ids = (~df["movie_id"].isin(valid_movie_ids)).sum()
    print(f"{name}: {invalid_ids} invalid movie IDs")

RELATIONSHIP VALIDATION
----------------------------------------
Movie Genres: 0 invalid movie IDs
Cast: 0 invalid movie IDs
Crew: 0 invalid movie IDs
Movie Keywords: 0 invalid movie IDs


In [17]:
valid_genre_ids = set(genres["genre_id"])

invalid_genres = (~movie_genres["genre_id"].isin(valid_genre_ids)).sum()

print("Invalid genre IDs:", invalid_genres)


Invalid genre IDs: 0


In [18]:
cleaned_files = {
    "movies_cleaned.csv": movies,
    "genres_cleaned.csv": genres,
    "movie_genres_cleaned.csv": movie_genres,
    "cast_cleaned.csv": cast,
    "crew_cleaned.csv": crew,
    "movie_keywords_cleaned.csv": movie_keywords
}

for filename, df in cleaned_files.items():
    save_path = data_folder / filename
    df.to_csv(save_path, index=False)
    print(f"Saved: {filename}")

Saved: movies_cleaned.csv
Saved: genres_cleaned.csv
Saved: movie_genres_cleaned.csv
Saved: cast_cleaned.csv
Saved: crew_cleaned.csv
Saved: movie_keywords_cleaned.csv


In [19]:
print("CLEANED FILE VERIFICATION")
print("-" * 35)

for filename in cleaned_files:
    file_path = data_folder / filename
    print(filename, "→", "Available" if file_path.exists() else "Missing")

CLEANED FILE VERIFICATION
-----------------------------------
movies_cleaned.csv → Available
genres_cleaned.csv → Available
movie_genres_cleaned.csv → Available
cast_cleaned.csv → Available
crew_cleaned.csv → Available
movie_keywords_cleaned.csv → Available


In [20]:
movie_file = data_folder / "movies_cleaned.csv"

print("File exists:", movie_file.exists())
print("File size:", movie_file.stat().st_size, "bytes")

check_movies = pd.read_csv(movie_file)

print("\nColumn names:")
print(check_movies.columns.tolist())

print("\nFirst 5 records:")
display(check_movies.head())

File exists: True
File size: 915215 bytes

Column names:
['movie_id', 'title', 'original_language', 'release_date', 'runtime', 'budget', 'revenue', 'popularity', 'vote_average', 'vote_count', 'status', 'overview']

First 5 records:


,movie_id,title,original_language,release_date,runtime,budget,revenue,popularity,vote_average,vote_count,status,overview
0,157336,Interstellar,en,2014-11-05,169,165000000.0,7.466067e+08,73.0452,8.484,40566,Released,The adventures of a group of explorers who mak...
1,27205,Inception,en,2010-07-15,148,160000000.0,8.390306e+08,55.8497,8.400,39727,Released,"Cobb, a skilled thief who commits corporate es..."
2,24428,The Avengers,en,2012-04-25,143,220000000.0,1.518816e+09,72.2515,8.062,38993,Released,When an unexpected enemy emerges and threatens...
3,155,The Dark Knight,en,2008-07-16,152,185000000.0,1.004558e+09,60.5510,8.534,36255,Released,Batman raises the stakes in his war on crime. ...
4,19995,Avatar,en,2009-12-16,162,237000000.0,2.923706e+09,48.1996,7.610,34348,Released,"In the 22nd century, a paraplegic Marine is di..."


In [21]:
import mysql.connector

print("MySQL connector is ready")

ModuleNotFoundError: No module named 'mysql'

In [22]:
%pip install mysql-connector-python

   ---------------------------------------- 0.0/17.7 MB ? eta -:--:--
    --------------------------------------- 0.3/17.7 MB ? eta -:--:--
   - -------------------------------------- 0.8/17.7 MB 2.5 MB/s eta 0:00:07
   -- ------------------------------------- 1.3/17.7 MB 2.2 MB/s eta 0:00:08
   --- ------------------------------------ 1.6/17.7 MB 2.1 MB/s eta 0:00:08
   ---- ----------------------------------- 2.1/17.7 MB 2.1 MB/s eta 0:00:08
   ----- ---------------------------------- 2.6/17.7 MB 2.1 MB/s eta 0:00:08
   ------ --------------------------------- 2.9/17.7 MB 2.1 MB/s eta 0:00:07
   ------- -------------------------------- 3.4/17.7 MB 2.1 MB/s eta 0:00:07
   -------- ------------------------------- 3.9/17.7 MB 2.2 MB/s eta 0:00:07
   ---------- ----------------------------- 4.5/17.7 MB 2.2 MB/s eta 0:00:07
   ----------- ---------------------------- 5.0/17.7 MB 2.2 MB/s eta 0:00:06
   ------------ --------------------------- 5.5/17.7 MB 2.2 MB/s eta 0:00:06
   ----------


[notice] A new release of pip is available: 26.2.2 -> 26.3
[notice] To update, run: python.exe -m pip install --upgrade pip


In [23]:
import mysql.connector

print("MySQL connector is ready!")

MySQL connector is ready!


In [25]:
import mysql.connector
from getpass import getpass

connection = mysql.connector.connect(
    host="localhost",
    user="root",
    password=getpass("Enter your MySQL password: "),
    database="tmdb_movie_analytics",
    port=3306
)

if connection.is_connected():
    print("Successfully connected to TMDB database!")

Successfully connected to TMDB database!


In [26]:
cursor = connection.cursor()

cursor.execute("SELECT DATABASE()")
print("Connected database:", cursor.fetchone()[0])

cursor.execute("SHOW TABLES")
print("Existing tables:", cursor.fetchall())

Connected database: tmdb_movie_analytics
Existing tables: []


In [27]:
create_movies_table = """
CREATE TABLE IF NOT EXISTS movies (
    movie_id INT PRIMARY KEY,
    title VARCHAR(500),
    original_language VARCHAR(10),
    release_date DATE,
    runtime INT,
    budget DOUBLE,
    revenue DOUBLE,
    popularity DOUBLE,
    vote_average DOUBLE,
    vote_count INT,
    status VARCHAR(30),
    overview TEXT
)
"""

cursor.execute(create_movies_table)
connection.commit()

print("Movies table created successfully!")

Movies table created successfully!


In [28]:
insert_query = """
INSERT INTO movies (
    movie_id, title, original_language, release_date,
    runtime, budget, revenue, popularity,
    vote_average, vote_count, status, overview
)
VALUES (%s, %s, %s, %s, %s, %s, %s, %s, %s, %s, %s, %s)
"""

movie_records = []

for row in movies.itertuples(index=False, name=None):
    values = []
    for value in row:
        if pd.isna(value):
            values.append(None)
        elif isinstance(value, pd.Timestamp):
            values.append(value.to_pydatetime())
        elif isinstance(value, np.generic):
            values.append(value.item())
        else:
            values.append(value)

    movie_records.append(tuple(values))

cursor.executemany(insert_query, movie_records)
connection.commit()

print("Movie records inserted:", cursor.rowcount)

Movie records inserted: 2503


In [29]:
cursor.execute("SELECT COUNT(*) FROM movies")

movie_count = cursor.fetchone()[0]

print("Total movies stored in MySQL:", movie_count)

Total movies stored in MySQL: 2503


In [30]:
cursor.execute("""
CREATE TABLE IF NOT EXISTS genres (
    genre_id INT PRIMARY KEY,
    genre_name VARCHAR(100)
)
""")

connection.commit()

print("Genres table created successfully!")

Genres table created successfully!


In [31]:
genre_query = """
INSERT INTO genres (genre_id, genre_name)
VALUES (%s, %s)
"""

genre_records = list(
    genres[["genre_id", "genre_name"]].itertuples(index=False, name=None)
)

cursor.executemany(genre_query, genre_records)
connection.commit()

print("Genre records inserted:", cursor.rowcount)

Genre records inserted: 19


In [32]:
cursor.execute("""
CREATE TABLE IF NOT EXISTS movie_genres (
    movie_id INT,
    genre_id INT,
    PRIMARY KEY (movie_id, genre_id),
    FOREIGN KEY (movie_id) REFERENCES movies(movie_id),
    FOREIGN KEY (genre_id) REFERENCES genres(genre_id)
)
""")

connection.commit()

print("Movie Genres table created successfully!")

Movie Genres table created successfully!


In [33]:
genre_link_query = """
INSERT INTO movie_genres (movie_id, genre_id)
VALUES (%s, %s)
"""

genre_link_records = list(
    movie_genres[["movie_id", "genre_id"]].itertuples(index=False, name=None)
)

cursor.executemany(genre_link_query, genre_link_records)
connection.commit()

print("Movie-genre records inserted:", cursor.rowcount)

Movie-genre records inserted: 6878


In [34]:
cursor.execute("""
CREATE TABLE IF NOT EXISTS cast (
    cast_row_id INT PRIMARY KEY,
    movie_id INT,
    person_id INT,
    actor_name VARCHAR(255),
    character_name TEXT,
    cast_order INT,
    FOREIGN KEY (movie_id) REFERENCES movies(movie_id)
)
""")

connection.commit()

print("Cast table created successfully!")

Cast table created successfully!


In [35]:
cast_query = """
INSERT INTO cast (
    cast_row_id, movie_id, person_id,
    actor_name, character_name, cast_order
)
VALUES (%s, %s, %s, %s, %s, %s)
"""

cast_records = list(
    cast[[
        "cast_row_id", "movie_id", "person_id",
        "actor_name", "character_name", "cast_order"
    ]].itertuples(index=False, name=None)
)

cursor.executemany(cast_query, cast_records)
connection.commit()

print("Cast records inserted:", cursor.rowcount)

Cast records inserted: 24902


In [36]:
cursor.execute("""
CREATE TABLE IF NOT EXISTS crew (
    crew_row_id INT PRIMARY KEY,
    movie_id INT,
    person_id INT,
    person_name VARCHAR(255),
    job VARCHAR(255),
    department VARCHAR(100),
    FOREIGN KEY (movie_id) REFERENCES movies(movie_id)
)
""")

connection.commit()

print("Crew table created successfully!")

Crew table created successfully!


In [37]:
crew_query = """
INSERT INTO crew (
    crew_row_id, movie_id, person_id,
    person_name, job, department
)
VALUES (%s, %s, %s, %s, %s, %s)
"""

crew_records = list(
    crew[[
        "crew_row_id", "movie_id", "person_id",
        "person_name", "job", "department"
    ]].itertuples(index=False, name=None)
)

cursor.executemany(crew_query, crew_records)
connection.commit()

print("Crew records inserted:", cursor.rowcount)

Crew records inserted: 7289


In [38]:
cursor.execute("""
CREATE TABLE IF NOT EXISTS movie_keywords (
    movie_id INT,
    keyword_id INT,
    keyword_name VARCHAR(255),
    PRIMARY KEY (movie_id, keyword_id),
    FOREIGN KEY (movie_id) REFERENCES movies(movie_id)
)
""")

connection.commit()

print("Movie Keywords table created successfully!")

Movie Keywords table created successfully!


In [39]:
keyword_query = """
INSERT INTO movie_keywords (
    movie_id, keyword_id, keyword_name
)
VALUES (%s, %s, %s)
"""

keyword_records = list(
    movie_keywords[
        ["movie_id", "keyword_id", "keyword_name"]
    ].itertuples(index=False, name=None)
)

cursor.executemany(keyword_query, keyword_records)
connection.commit()

print("Keyword records inserted:", cursor.rowcount)

Keyword records inserted: 41681


In [40]:
table_names = [
    "movies",
    "genres",
    "movie_genres",
    "cast",
    "crew",
    "movie_keywords"
]

print("FINAL MYSQL TABLE VERIFICATION")
print("-" * 45)

for table in table_names:
    cursor.execute(f"SELECT COUNT(*) FROM `{table}`")
    count = cursor.fetchone()[0]
    print(f"{table:<20} : {count}")

FINAL MYSQL TABLE VERIFICATION
---------------------------------------------
movies               : 2503
genres               : 19
movie_genres         : 6878
cast                 : 24902
crew                 : 7289
movie_keywords       : 41681
